In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"  # Ajustar se necessário

# ========== FILTRO DE RESTAURANTES E ESTABELECIMENTOS DE COMIDA ==========

FOOD_TAGS = {
    "Restaurants", "Food", "Fast Food", "Food Trucks", "Food Delivery Services",
    "Pizza", "Mexican", "Chinese", "Italian", "Japanese", "Thai", "Vietnamese",
    "Indian", "Greek", "Mediterranean", "French", "Korean", "Filipino", "African",
    "Cuban", "Caribbean", "Middle Eastern", "Latin American", "Asian Fusion",
    "American (Traditional)", "American (New)", "Canadian (New)", "Cajun/Creole",
    "Pakistani", "Southern", "Soul Food", "Tex-Mex",
    "Burgers", "Sandwiches", "Sushi Bars", "Seafood", "Steakhouses", "Barbeque",
    "Chicken Wings", "Chicken Shop", "Hot Dogs", "Cheesesteaks", "Tacos",
    "Delis", "Diners", "Buffets", "Breakfast & Brunch",
    "Cafes", "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
    "Juice Bars & Smoothies", "Desserts", "Bakeries", "Donuts", "Bagels",
    "Ice Cream & Frozen Yogurt", "Candy Stores",
    "Bars", "Breweries", "Wine & Spirits", "Beer", "Cocktail Bars",
    "Dive Bars", "Sports Bars", "Pubs", "Gastropubs", "Lounges",
    "Grocery", "Specialty Food", "Seafood Markets", "Meat Shops", "Fruits & Veggies",
    "Farmers Market", "Convenience Stores", "Wholesale Stores",
    "Caterers",
}

def is_food_related(categories_str):
    """Verifica se o estabelecimento é relacionado a alimentação."""
    if not categories_str:
        return False
    tags = {t.strip() for t in str(categories_str).split(",")}
    return bool(tags & FOOD_TAGS)

def get_segmento_alimentacao(categories_str):
    """Retorna o segmento de alimentação baseado nas categorias."""
    if not categories_str:
        return "Outros alimentação"
    
    tags = {t.strip() for t in str(categories_str).split(",")}
    
    if "Restaurants" in tags:
        return "RESTAURANTE"
    if tags & {"Bars", "Breweries", "Cocktail Bars", "Dive Bars",
                "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer"}:
        return "BAR E BEBIDA"
    if tags & {"Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
                "Cafes", "Juice Bars & Smoothies"}:
        return "CAFE"
    if tags & {"Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
                "Candy Stores", "Desserts"}:
        return "PADARIA"
    segmentos_alimentacao = {
        "Restaurants",
        "Bars", "Breweries", "Cocktail Bars", "Dive Bars",
        "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer",
        "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
        "Cafes", "Juice Bars & Smoothies",
        "Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
        "Candy Stores", "Desserts"
    }
    if not tags & segmentos_alimentacao:
        return "OUTROS"

is_food_udf = udf(is_food_related, BooleanType())
get_segmento_udf = udf(get_segmento_alimentacao, StringType())

def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    total_registros = df.count()
    metricas = {}
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    condicao = col(coluna).isNotNull()
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))



In [0]:

# COLOCAR LIMITE DA BASE SE QUISER AMOSTRA ==============================================================================================================
from pyspark.sql.functions import upper, regexp_replace, trim

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"  # Ajustar se necessário

# ========== TABELA 1: BUSINESS ==========

table_name = "bronze_yelp_academic_dataset_business"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_business = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_business.count()}")

# ========== FILTRO: ESTABELECIMENTOS DE ALIMENTAÇÃO ==========
print("\n--- Filtro de Estabelecimentos de Alimentação ---")
df_business_food = df_business.filter(is_food_udf(col('categories')))
print(f"Registros relacionados a alimentação: {df_business_food.count()}")

# Adiciona coluna de food_category
df_business_food = df_business_food.withColumn('food_category', get_segmento_udf(col('categories')))
print("✓ Coluna 'food_category' adicionada")

# Filtra para não incluir 'OUTROS' na silver
df_business_food = df_business_food.filter(col('food_category') != "OUTROS")
print("✓ Registros com food_category = 'OUTROS' removidos")

# Mostra distribuição por food_category
print("\nDistribuição por food_category:")
display(df_business_food.groupBy('food_category').count().orderBy(col('count').desc()))

# Remove registros com is_open igual a 0 -> estão fechados permanentemente
df_business_food = df_business_food.filter(col('is_open') != 0)
print(f"✓ Registros após remoção de is_open = 0: {df_business_food.count()} ")

# Remove colunas address, attributes, categories, is_open, latitude, longitude, postal_code que não serão utilizadas
df_business_food = df_business_food.drop(
    'address', 'attributes', 'categories', 'is_open', 'latitude', 'longitude', 'postal_code'
)
print("✓ Colunas removidas: address, attributes, categories, is_open, latitude, longitude, postal_code")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'name', 'city', 'state']
metricas_completude = calcular_completude(df_business_food, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_business_food_clean = df_business_food.filter(
    col('business_id').isNotNull() & 
    col('name').isNotNull() & 
    col('city').isNotNull() & 
    col('state').isNotNull()
)

print(f"\nApós filtro de completude: {df_business_food_clean.count()} registros")

# PRECISÃO: Validação de ranges numéricos
print("\n--- Validação de PRECISÃO ---")

# Stars: 0 a 5
df_business_food_clean = validar_precisao_numerica(df_business_food_clean, 'stars', min_val=0, max_val=5)
print(f"  Stars (0-5): {df_business_food_clean.count()} registros válidos")

# Review count: >= 0
df_business_food_clean = df_business_food_clean.filter(col('review_count') >= 0)
print(f"  Review count (>=0): {df_business_food_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_business_food_clean = remover_duplicados(df_business_food_clean, ['business_id'])
print(f"Após deduplicação: {df_business_food_clean.count()} registros")

# ========== PADRONIZAÇÃO DO NOME ==========
print("\n--- Padronização de Nomes ---")

# Apply transformations step by step
name_col = upper(col("name"))
name_col = regexp_replace(name_col, r",.*", "")  # Remove text after comma
name_col = regexp_replace(name_col, r",", " ")  # Replace comma with space
name_col = regexp_replace(name_col, r"\.", " ")  # Replace dot with space
name_col = regexp_replace(name_col, r"ST\.", "SAINT")
name_col = regexp_replace(name_col, r"\bST\b", "SAINT")
name_col = regexp_replace(name_col, r"\bST\.\b", "SAINT")
name_col = regexp_replace(name_col, r"SAINTT", "SAINT")
name_col = regexp_replace(name_col, r"NW ", "NEW")
name_col = regexp_replace(name_col, r"'", " ")  # Replace apostrophes
name_col = regexp_replace(name_col, r"-", " ")  # Replace hyphens
name_col = regexp_replace(name_col, r"^ +| +$", "")  # Trim edges
name_col = regexp_replace(name_col, r" {2,}", " ")  # Collapse multiple spaces
name_col = trim(name_col)

df_business_food_clean = df_business_food_clean.withColumn("name_validated", name_col)
print("✓ Coluna 'name_validated' adicionada")

# ========== PADRONIZAÇÃO DA CIDADE ==========
print("\n--- Padronização de Cidades ---")

# Apply transformations step by step
city_col = upper(col("city"))
city_col = regexp_replace(city_col, r",.*", "")  # Remove content after comma
city_col = regexp_replace(city_col, r"/.*", "")  # Remove content after slash
city_col = regexp_replace(city_col, r"BCH", "BEACH")
city_col = regexp_replace(city_col, r",", " ")
city_col = regexp_replace(city_col, r"/", " ")
city_col = regexp_replace(city_col, r"%MTLAUREL%", "MT LAUREL")
city_col = regexp_replace(city_col, r"%TAMPA FLORIDA%", "TAMPA")
city_col = regexp_replace(city_col, r"TWP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"MT \.", "MT")
city_col = regexp_replace(city_col, r"MT\.", "MT")
city_col = regexp_replace(city_col, r"SAINTLOUIS", "SAINT LOUIS")
city_col = regexp_replace(city_col, r"SAINTT", "SAINT")
city_col = regexp_replace(city_col, r"^SAINT PETE$", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"SAINT PETERS", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"REDINGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"REDNGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"CHALEMETTE", "CHALMETTE")
city_col = regexp_replace(city_col, r"INPOLIS", "INDIANAPOLIS")
city_col = regexp_replace(city_col, r"CONSHOHOEKEN", "CONSHOHOCKEN")
city_col = regexp_replace(city_col, r"FESTERVILLE", "FEASTERVILLE")
city_col = regexp_replace(city_col, r"TOWSSHIP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"%TWN%", "TOWN")
city_col = regexp_replace(city_col, r"CNTRY", "COUNTRY")
city_col = regexp_replace(city_col, r"TIERRE VERDE", "TIERRA VERDE")
city_col = regexp_replace(city_col, r"NW", "NEW")
city_col = regexp_replace(city_col, r"TOWN & COUNTRY", "TOWN N COUNTRY")
city_col = regexp_replace(city_col, r"\.", " ")  # Replace dot with space
city_col = regexp_replace(city_col, r"ST\.", "SAINT")
city_col = regexp_replace(city_col, r"\bST\b", "SAINT")
city_col = regexp_replace(city_col, r"\bST\.\b", "SAINT")
city_col = regexp_replace(city_col, r"'", " ")  # Replace apostrophes
city_col = regexp_replace(city_col, r"-", " ")  # Replace hyphens
city_col = regexp_replace(city_col, r" {2,}", " ")  # Collapse multiple spaces
city_col = trim(city_col)  # Remove spaces at the beginning and end

df_business_food_clean = df_business_food_clean.withColumn("city_validated", city_col)
print("✓ Coluna 'city_validated' adicionada")

# ========== SUBSTITUIÇÃO DOS CAMPOS ORIGINAIS ==========
print("\n--- Substituição de Campos Originais ---")

# Remove campos originais name e city
df_business_food_clean = df_business_food_clean.drop('name', 'city')
print("✓ Campos originais 'name' e 'city' removidos")

# Renomeia os campos validados
df_business_food_clean = df_business_food_clean.withColumnRenamed('name_validated', 'name')
df_business_food_clean = df_business_food_clean.withColumnRenamed('city_validated', 'city')
print("✓ Campos renomeados: 'name_validated' -> 'name', 'city_validated' -> 'city'")

# Adiciona metadados silver
df_business_silver = adicionar_metadados_silver(df_business_food_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.new_silver_business"
df_business_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_business_silver.count()}")
print("="*60)
display(spark.table(table_silver).limit(20))

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Business:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.new_silver_business")

# Métricas gerais
print(f"Total de registros: {df_sample.count()}")
print(f"Campos: {len(df_sample.columns)}")

# Amostra de dados
print("\nPrimeiros 10 registros:")
display(df_sample.select(
    'business_id', 
    'name',
    'city',
    'state', 
    'stars', 
    'review_count',
    'food_category',
    'data_processamento_silver'
).limit(10))

# Estatísticas de qualidade
print("\nEstatísticas de Stars:")
df_sample.select('stars').describe().show()

print("\nDistribuição por Estado (Top 10):")
display(df_sample.groupBy('state').count().orderBy(col('count').desc()).limit(10))